# Testing DAG Airflow Version - Take#1
### TAM Penetration Counts - Pharos SQL Testing
Based on the Airflow DAG version of tam_penetration_counts_dag.py. Uses Jinja2 render_sql() and run_cli() helper.

In [ ]:
# coding=utf-8
import os
import json
import subprocess as sp
from io import StringIO
import pandas as pd
from jinja2 import Environment, FileSystemLoader

# Airflow Imports (commented out for notebook usage)
# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from airflow.utils.email import send_email
# import pendulum

In [ ]:
# --- Configuration & Constants ---
DAG_HOME = os.path.dirname(os.path.abspath("__file__"))
main_table_name = "tam_penetration_counts"

email_list = [
    "huzefa.saifee@workday.com",
    "m6a0l2y5u3c9i6f3@workday.enterprise.slack.com",
]

In [ ]:
def run_cli(cmd, fetch_data=False):
    """
    Executes a pharos CLI command.
    If fetch_data=True, parses the stdout as JSON and returns the result.data (CSV).
    """
    result = sp.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {result.returncode}\nCMD: {cmd}\nSTDERR: {result.stderr}"
        )

    raw_output = result.stdout.strip()
    if fetch_data:
        try:
            parsed = json.loads(raw_output)
            return parsed["result"]["data"]
        except json.JSONDecodeError as e:
            raise RuntimeError(
                f"Failed to parse JSON output. Command: {cmd}\nOutput: {raw_output[:300]}"
            ) from e
    return raw_output

In [ ]:
def render_sql(filename, **kwargs):
    """Loads and renders a Jinja2 template SQL file dynamically."""
    env = Environment(loader=FileSystemLoader(DAG_HOME))
    template = env.get_template(filename)
    return template.render(**kwargs)

In [ ]:
def build_insert_query(select_sql):
    """Assemble: INSERT INTO + WITH + CTEs (raw text) + SELECT (rendered)."""
    cte_path = os.path.join(DAG_HOME, "penetration_ctes.sql")
    with open(cte_path, "r") as f:
        cte_sql = f.read()
    return f"INSERT INTO cdt.{main_table_name}\nWITH\n{cte_sql}\n{select_sql}"

In [ ]:
# --- Ensure Table Exists ---
create_ddl = render_sql("create_table.sql", table_name=main_table_name)
create_cmd = f'pharos sql run --sql "{create_ddl}"'

tables_csv = run_cli('pharos sql run --sql "SHOW TABLES IN dw.cdt"', fetch_data=True)
tables = tables_csv.split("\n")

if main_table_name not in tables:
    print(f"Table cdt.{main_table_name} not found. Creating...")
    run_cli(create_cmd, fetch_data=False)
else:
    try:
        run_cli(
            f'pharos sql run --sql "SELECT COUNT(*) FROM cdt.{main_table_name} WHERE snapshot_date IS NOT NULL LIMIT 1"',
            fetch_data=True,
        )
        print(f"Table cdt.{main_table_name} exists and is healthy.")
    except RuntimeError:
        print(f"Table cdt.{main_table_name} is broken. Dropping and recreating...")
        run_cli(
            f'pharos sql run --sql "DROP TABLE IF EXISTS cdt.{main_table_name}"',
            fetch_data=False,
        )
        run_cli(create_cmd, fetch_data=False)

In [ ]:
# --- Delete today's partition (idempotent overwrite) ---
pacific_today = "CAST(CURRENT_TIMESTAMP AT TIME ZONE 'America/Los_Angeles' AS DATE)"
print(f"Deleting existing partition for today (Pacific)...")
run_cli(
    f'pharos sql run --sql "DELETE FROM cdt.{main_table_name} WHERE snapshot_date = {pacific_today}"',
    fetch_data=False,
)
print("Done.")

In [ ]:
# --- Compute & Insert daily snapshot ---
select_sql = render_sql("compute_select.sql", comments="Live daily snapshot")
full_query = build_insert_query(select_sql)

print("Executing penetration counts query...")
run_cli(f'pharos sql run --sql "{full_query}"', fetch_data=False)
print("Daily snapshot inserted successfully.")

In [ ]:
# --- Validate: Fetch summary from the table ---
segment_csv = run_cli(
    f'pharos sql run --sql "SELECT market_segment, COUNT(*) AS row_count, MIN(snapshot_date) AS first_date, MAX(snapshot_date) AS last_date FROM cdt.{main_table_name} WHERE snapshot_date IS NOT NULL GROUP BY market_segment ORDER BY market_segment"',
    fetch_data=True,
)
df_segments = pd.read_csv(StringIO(segment_csv))
print(f"Segment breakdown:")
df_segments

In [ ]:
# --- Fetch today's data for inspection ---
today_csv = run_cli(
    f'pharos sql run --sql "SELECT * FROM cdt.{main_table_name} WHERE snapshot_date = {pacific_today}"',
    fetch_data=True,
)
df = pd.read_csv(StringIO(today_csv))
print(f"Today's rows: {len(df)}")
df